In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
# Load a fresh tracer process and the supplied delay calculation.
%run ../resources/optimization_setup.py

# 99.5 Advanced Optimization

## Contents

**Tools:** CADET-Process (Python).

**Python used:** Parameter paths, objective functions, simulation results and callbacks.

**Introduces:** variable targets, evaluators, normalization, `COBYLA`, multiple evaluation cases and shared evaluation.

1. Connect a variable to a process
2. Simulate and evaluate one candidate
3. Normalize and optimize
4. Reuse the setup for multiple experiments

In the RTD lesson, the bypass experiment revealed the delay added by the mixer and tubing. Here, choose a tubing length that gives the setup a **40 s mean delay** at **6 mL/min**.

The dedicated optimization module connects each candidate to a chain of model changes, simulations and metrics. Shared dependencies and cached results avoid repeating work for every objective or callback. The problem definition also stays the same when switching algorithms.

```{dropdown} Competing objectives and choosing a result
For conflicting goals, such as high productivity and low eluent consumption, a multi-objective optimizer can return several trade-offs. A Pareto result cannot improve one objective without worsening another. Purity constraints first rule out unacceptable products; the remaining choice depends on the process priorities. There need not be one best setting for every goal.
```

## Start from the tracer experiment

The supplied process contains the familiar **1 mL mixer**, **2 mL tubing at 1 m length**, and a **5 s tracer pulse**. Its initial mean delay is about **30 s**.

The mean delay is the outlet pulse's concentration-weighted mean time minus the inlet pulse's mean time. The shared helper performs that calculation in seconds.

In [ ]:
# Create and simulate the baseline before defining the optimization.
from CADETProcess.simulator import Cadet

tracer_process = tracer_delay_process("delay_design")
simulator = Cadet()
baseline_results = simulator.simulate(tracer_process)
print(f"Baseline mean delay: {transport_delay(baseline_results):.1f} s")
_ = baseline_results.solution.outlet.outlet.plot()
plt.show()

## Connect the variable to the process

The process is registered as an **evaluation object**: a case to run for each candidate. The variable's `targets` select the object to change; `parameter_path` identifies the parameter inside it.

Vary tubing length between **0.25 and 2 m**, keeping diameter and flow rate fixed.

In [ ]:
from CADETProcess.optimization import OptimizationProblem

delay_problem = OptimizationProblem("tracer_delay")
delay_problem.add_evaluation_object(tracer_process)

In [ ]:
delay_problem.add_variable(
    "tubing_length", targets=[tracer_process],
    parameter_path="flow_sheet.tubing.length",
    lb=0.25, ub=2.0, normalization="linear",
)

```{note}
The optimizer chooses `tubing_length`. CADET-Process writes that value into `tracer_process.flow_sheet.tubing.length` before simulation. Bounds and candidate values are still expressed in metres.
```

## From a simulation to an objective

`Cadet` is registered as an **evaluator**. The objective receives its simulation results and returns the squared difference from the desired **40 s** delay.

**Length → process → simulation → delay → squared error**

`requires=[simulator]` tells CADET-Process to run the simulator before evaluating this objective.

In [ ]:
delay_problem.add_evaluator(simulator)

In [ ]:
def delay_error(results):
    return (transport_delay(results) - 40) ** 2

delay_problem.add_objective(
    delay_error, name="delay_error", requires=[simulator],
)

At **1 m**, the delay is about **30 s**, so the error should be about **100 s²**. We evaluate this candidate before launching the search.

In [ ]:
print("Objective at 1 m:", delay_problem.evaluate_objectives([1.0]))

## Normalize and optimize

Linear normalization maps the allowed range to **0–1** internally. This helps when variables have different scales, such as lengths in metres and flow rates in m³/s.

The public inputs and results retain their physical units. Here, `x0=[1.0]` means **1 m**. `COBYLA` is a local, derivative-free optimizer, suitable for this simulated objective.

In [ ]:
from CADETProcess.optimization import COBYLA

delay_optimizer = COBYLA()
delay_optimizer.n_max_iter = 60
delay_results = delay_optimizer.optimize(delay_problem, x0=[1.0])
print("Best tubing length / m:", delay_results.x)
print("Squared delay error / s²:", delay_results.f)
_ = delay_results.plot_convergence("objectives")
plt.show()

## Check the selected process

At fixed diameter and flow, a longer tube adds more liquid volume and delay. A length near **1.5 m** gives **3 mL tubing + 1 mL mixer**, corresponding to **40 s** at **6 mL/min**.

The selected value is simulated explicitly to show the final chromatogram. The process may otherwise contain the last candidate the optimizer evaluated.

In [ ]:
# Apply the selected value and verify its delay with a new simulation.
tracer_process.flow_sheet.tubing.length = float(delay_results.x[0, 0])
selected_results = simulator.simulate(tracer_process)
print(f"Selected mean delay: {transport_delay(selected_results):.1f} s")
_ = selected_results.solution.outlet.outlet.plot()
plt.show()

```{dropdown} Other normalization choices
`normalization="log"` suits a positive parameter spanning several orders of magnitude, for example `lb=1e-9, ub=1e-5` for axial dispersion. `normalization="auto"` selects a suitable scaling from the bounds. Normalization changes the search coordinates; it does not change the physical model.
```

## Multiple experiments, one shared parameter

The pulse experiment is repeated with **5 s** and **10 s** injections. Both use the same mixer, tubing diameter and flow. After subtracting the inlet pulse's mean time, both should give the same transport delay.

Two fresh processes are created, and one tubing-length variable targets both. By default, the objective returns **one metric per case**.

In [ ]:
# Build independent cases rather than reusing the same process object.
short_pulse = tracer_delay_process("pulse_5s", pulse_length=5)
long_pulse = tracer_delay_process("pulse_10s", pulse_length=10)
joint_problem = OptimizationProblem("two_pulses")
joint_problem.add_evaluation_object(short_pulse)
joint_problem.add_evaluation_object(long_pulse)
joint_problem.add_variable(
    "tubing_length", targets=[short_pulse, long_pulse],
    parameter_path="flow_sheet.tubing.length",
    lb=0.25, ub=2.0, normalization="linear",
)
joint_problem.add_evaluator(simulator)
joint_problem.add_objective(delay_error, requires=[simulator])
print("Separate case errors at 1.5 m:", joint_problem.evaluate_objectives([1.5]))

```{dropdown} One combined metric
For a joint fit, `per_object=False` passes the case results together to an objective. That function must combine them, for example by adding their squared errors. Separate metrics retain each experiment's contribution; a combined metric gives the optimizer one score.
```

## Optional: show progress with a callback

A callback can plot a chromatogram for the selected candidates during optimization. It uses the same simulation dependency as the objective, so shared results can be reused rather than simulated separately for each metric.

The callback is registered before the optimizer runs. It receives the simulation result and an `IndividualView` with the candidate's named values.

In [ ]:
def plot_progress(results, individual, evaluation_object, callbacks_dir):
    _ = results.solution.outlet.outlet.plot(
        file_name=f"{callbacks_dir}/{individual.id}_{evaluation_object.name}.png",
    )

# Register this before rerunning the optimizer if progress plots are wanted.
delay_problem.add_callback(plot_progress, requires=[simulator], frequency=5)

```{dropdown} How the optimization suite is organized
`OptimizationProblem` is the entry point used throughout this workshop. It coordinates three reusable parts:

| Part | Responsibility |
|---|---|
| `ParameterSpace` | Variables, bounds, targets and normalization |
| `EvaluationPipeline` | Dependencies, execution of cases and cached intermediate results |
| `MetricSpace` | Named objectives, directions and constraint bounds |

`Problem` combines these for evaluation and exploration without running an optimizer. The workshop uses the higher-level `OptimizationProblem` methods to keep process setup and optimization in one place.
```

## Summary

- `targets` and `parameter_path` connect variables to process attributes.
- Evaluator dependencies connect simulations, metrics and callbacks.
- Several experiments can share a parameter while retaining separate comparison metrics.

**Now:** **99.5 Advanced Optimization: Exercises**: fit binding parameters to three experiments with one shared parameter set.